# 07 — Final Untouched Test Evaluation

The complete development process is now frozen.

This notebook performs the **single final evaluation** on the untouched final 15% of the chronological UFC dataset.

It does **not** perform feature engineering, Elo tuning, CatBoost hyperparameter tuning, calibration-method selection, or threshold optimization.

The notebook loads:

- `frozen_catboost_params.json`
- `frozen_calibration_decision.json`
- `frozen_feature_manifest.json`

It then fits the final model on development data, classifies the untouched test set, evaluates the final probabilities, and saves the final artifacts.

After this notebook is run, I should **not modify the model based on the final test result**.

## Final evaluation protocol

The chronological split is fixed:

```text
0% ------------------------------------------------------------ 100%

|<------------------- DEVELOPMENT ------------------>|<- TEST ->|
0%                                                   85%        100%
```

If the frozen calibration method is `platt` or `isotonic`, the final calibration mapping is fitted using only **chronological out-of-fold development predictions**.

Example:

```text
train 0–50%   → predict 50–60%
train 0–60%   → predict 60–70%
train 0–70%   → predict 70–77.5%
train 0–77.5% → predict 77.5–85%
```

The final CatBoost model is then trained on the complete 0–85% development interval and evaluated on the untouched 85–100% test interval.

In [ ]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    log_loss,
    roc_auc_score,
)

RANDOM_SEED = 42

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)


## Project paths and frozen artifacts

In [ ]:
def find_project_root() -> Path:
    starts = [Path.cwd(), Path.cwd().parent]

    for start in starts:
        for path in [start, *start.parents]:
            if (path / "src").exists() and (path / "data").exists():
                return path

    raise FileNotFoundError("Could not locate the UFC_PREDICTOR project root.")


PROJECT_ROOT = find_project_root()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ufcstats"
MODEL_RESULTS_DIR = PROJECT_ROOT / "reports" / "final_model_optimization"
FINAL_RESULTS_DIR = PROJECT_ROOT / "reports" / "final_test"
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROCESSED_DIR / "fight_dataset.parquet"

FROZEN_PARAMS_PATH = MODEL_RESULTS_DIR / "frozen_catboost_params.json"
FROZEN_CALIBRATION_PATH = MODEL_RESULTS_DIR / "frozen_calibration_decision.json"
FROZEN_MANIFEST_PATH = MODEL_RESULTS_DIR / "frozen_feature_manifest.json"

FINAL_MODEL_PATH = FINAL_RESULTS_DIR / "final_catboost_model.cbm"
FINAL_CALIBRATOR_PATH = FINAL_RESULTS_DIR / "final_calibrator.pkl"
FINAL_PREDICTIONS_PATH = FINAL_RESULTS_DIR / "final_test_predictions.csv"
FINAL_METRICS_PATH = FINAL_RESULTS_DIR / "final_test_metrics.json"
FINAL_RELIABILITY_PATH = FINAL_RESULTS_DIR / "final_test_reliability.csv"

required_paths = [
    DATASET_PATH,
    FROZEN_PARAMS_PATH,
    FROZEN_CALIBRATION_PATH,
    FROZEN_MANIFEST_PATH,
]

missing = [path for path in required_paths if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Required frozen artifacts are missing:\n"
        + "\n".join(f"  - {path}" for path in missing)
    )

print("Project root:", PROJECT_ROOT)
print("Frozen artifacts:", MODEL_RESULTS_DIR)
print("Final results:", FINAL_RESULTS_DIR)


# Load the frozen configuration

In [ ]:
with FROZEN_PARAMS_PATH.open("r", encoding="utf-8") as f:
    FROZEN_CATBOOST_PARAMS = json.load(f)

with FROZEN_CALIBRATION_PATH.open("r", encoding="utf-8") as f:
    FROZEN_CALIBRATION = json.load(f)

with FROZEN_MANIFEST_PATH.open("r", encoding="utf-8") as f:
    FROZEN_FEATURE_MANIFEST = json.load(f)

FROZEN_CALIBRATION_METHOD = FROZEN_CALIBRATION["method"]

numeric_features = FROZEN_FEATURE_MANIFEST["numeric_model_features"]
categorical_features = FROZEN_FEATURE_MANIFEST["categorical_model_features"]
model_features = FROZEN_FEATURE_MANIFEST["all_model_features"]

print("Frozen CatBoost parameters:")
print(json.dumps(FROZEN_CATBOOST_PARAMS, indent=2))

print()
print("Frozen calibration method:", FROZEN_CALIBRATION_METHOD)
print("Frozen model features:", len(model_features))


# Load the production dataset and recreate the chronological split

In [ ]:
df = pd.read_parquet(DATASET_PATH)

df["event_date"] = pd.to_datetime(df["event_date"])

df = (
    df.sort_values(
        ["event_date", "event_id", "fight_order"],
        ascending=[True, True, False],
        kind="stable",
    )
    .reset_index(drop=True)
)

missing_features = set(model_features) - set(df.columns)

if missing_features:
    raise ValueError(
        "The production dataset does not match the frozen feature manifest. "
        f"Missing features: {sorted(missing_features)}"
    )

forbidden_features = [
    col
    for col in model_features
    if ("effective" in col or "elo_decay_fraction" in col)
]

if forbidden_features:
    raise ValueError(
        "Forbidden Effective/decay Elo features found in the frozen manifest:\n"
        + "\n".join(forbidden_features)
    )

if df["fight_id"].duplicated().any():
    raise ValueError("fight_dataset.parquet contains duplicate fight_id values.")

n = len(df)
development_end = int(n * 0.85)

development_idx = np.arange(0, development_end)
test_idx = np.arange(development_end, n)

split_summary = pd.DataFrame([
    {
        "split": "development",
        "rows": len(development_idx),
        "date_min": df.iloc[development_idx]["event_date"].min(),
        "date_max": df.iloc[development_idx]["event_date"].max(),
        "target_rate": df.iloc[development_idx]["target"].mean(),
    },
    {
        "split": "FINAL_TEST",
        "rows": len(test_idx),
        "date_min": df.iloc[test_idx]["event_date"].min(),
        "date_max": df.iloc[test_idx]["event_date"].max(),
        "target_rate": df.iloc[test_idx]["target"].mean(),
    },
])

display(split_summary)


## Prepare the frozen feature matrix

In [ ]:
def prepare_catboost_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame[model_features].copy()

    for col in categorical_features:
        frame[col] = (
            frame[col]
            .astype("string")
            .fillna("Unknown")
            .astype(str)
        )

    return frame


X_all = prepare_catboost_frame(df)
y_all = df["target"].astype(int)

X_development = X_all.iloc[development_idx]
y_development = y_all.iloc[development_idx]

X_test = X_all.iloc[test_idx]
y_test = y_all.iloc[test_idx]


# Metric helpers

In [ ]:
def expected_calibration_error(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 10,
) -> float:
    y_true = np.asarray(y_true, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probabilities, edges[1:-1], right=True)

    ece = 0.0

    for bin_id in range(n_bins):
        mask = bin_ids == bin_id

        if not mask.any():
            continue

        mean_probability = probabilities[mask].mean()
        observed_rate = y_true[mask].mean()

        ece += mask.mean() * abs(observed_rate - mean_probability)

    return float(ece)


def evaluate_probabilities(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-7,
        1 - 1e-7,
    )

    predictions = (probabilities >= 0.5).astype(int)

    return {
        "log_loss": float(log_loss(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "ece_10": expected_calibration_error(
            y_true,
            probabilities,
            n_bins=10,
        ),
    }


def probability_to_logit(probabilities: np.ndarray) -> np.ndarray:
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )

    return np.log(probabilities / (1.0 - probabilities))


# Fit the final calibration mapping from chronological OOF development predictions

This section is skipped if the frozen calibration method is `none`.

The calibration **method** is not selected here. It was already frozen in notebook 06.

In [ ]:
FINAL_CALIBRATOR = None
oof_calibration_df = None

if FROZEN_CALIBRATION_METHOD != "none":
    oof_folds = [
        {
            "name": "oof_1",
            "train_end": int(n * 0.50),
            "validation_start": int(n * 0.50),
            "validation_end": int(n * 0.60),
        },
        {
            "name": "oof_2",
            "train_end": int(n * 0.60),
            "validation_start": int(n * 0.60),
            "validation_end": int(n * 0.70),
        },
        {
            "name": "oof_3",
            "train_end": int(n * 0.70),
            "validation_start": int(n * 0.70),
            "validation_end": int(n * 0.775),
        },
        {
            "name": "oof_4",
            "train_end": int(n * 0.775),
            "validation_start": int(n * 0.775),
            "validation_end": development_end,
        },
    ]

    oof_rows = []

    for fold in oof_folds:
        train_idx = np.arange(0, fold["train_end"])
        validation_idx = np.arange(
            fold["validation_start"],
            fold["validation_end"],
        )

        print(
            f"Generating {fold['name']}: "
            f"{len(train_idx):,} train / "
            f"{len(validation_idx):,} validation"
        )

        fold_model = CatBoostClassifier(**FROZEN_CATBOOST_PARAMS)

        fold_model.fit(
            X_all.iloc[train_idx],
            y_all.iloc[train_idx],
            cat_features=categorical_features,
            verbose=False,
        )

        fold_probabilities = fold_model.predict_proba(
            X_all.iloc[validation_idx]
        )[:, 1]

        for local_index, row_index in enumerate(validation_idx):
            oof_rows.append({
                "row_index": int(row_index),
                "fold": fold["name"],
                "target": int(y_all.iloc[row_index]),
                "raw_probability": float(
                    fold_probabilities[local_index]
                ),
            })

    oof_calibration_df = (
        pd.DataFrame(oof_rows)
        .sort_values("row_index")
        .reset_index(drop=True)
    )

    print("OOF calibration rows:", f"{len(oof_calibration_df):,}")

    if FROZEN_CALIBRATION_METHOD == "platt":
        FINAL_CALIBRATOR = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        )

        FINAL_CALIBRATOR.fit(
            probability_to_logit(
                oof_calibration_df["raw_probability"].to_numpy()
            ).reshape(-1, 1),
            oof_calibration_df["target"].to_numpy(),
        )

    elif FROZEN_CALIBRATION_METHOD == "isotonic":
        FINAL_CALIBRATOR = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            out_of_bounds="clip",
        )

        FINAL_CALIBRATOR.fit(
            oof_calibration_df["raw_probability"].to_numpy(),
            oof_calibration_df["target"].to_numpy(),
        )

    else:
        raise ValueError(
            "Unknown frozen calibration method: "
            f"{FROZEN_CALIBRATION_METHOD}"
        )

else:
    print(
        "Frozen calibration method is 'none'. "
        "No final calibrator is fitted."
    )


# Train the final CatBoost model

In [ ]:
final_model = CatBoostClassifier(**FROZEN_CATBOOST_PARAMS)

final_model.fit(
    X_development,
    y_development,
    cat_features=categorical_features,
    verbose=False,
)

final_model.save_model(str(FINAL_MODEL_PATH))

print(
    "Final model trained on",
    f"{len(development_idx):,}",
    "development fights.",
)
print("Saved model:", FINAL_MODEL_PATH)


# Open the untouched final test set

In [ ]:
p_test_raw = final_model.predict_proba(X_test)[:, 1]

if FROZEN_CALIBRATION_METHOD == "none":
    p_test_final = p_test_raw.copy()

elif FROZEN_CALIBRATION_METHOD == "platt":
    p_test_final = FINAL_CALIBRATOR.predict_proba(
        probability_to_logit(
            p_test_raw
        ).reshape(-1, 1)
    )[:, 1]

elif FROZEN_CALIBRATION_METHOD == "isotonic":
    p_test_final = FINAL_CALIBRATOR.predict(p_test_raw)

else:
    raise ValueError("Unknown frozen calibration method.")

raw_test_metrics = evaluate_probabilities(
    y_test.to_numpy(),
    p_test_raw,
)

final_test_metrics = evaluate_probabilities(
    y_test.to_numpy(),
    p_test_final,
)

metrics_comparison = pd.DataFrame([
    {
        "probabilities": "raw_catboost",
        **raw_test_metrics,
    },
    {
        "probabilities": (
            "final"
            if FROZEN_CALIBRATION_METHOD == "none"
            else f"final_{FROZEN_CALIBRATION_METHOD}"
        ),
        **final_test_metrics,
    },
])

display(metrics_comparison)


## Final fight classifications

In [ ]:
test_output = df.iloc[test_idx][
    [
        "fight_id",
        "event_id",
        "event_name",
        "event_date",
        "weight_class",
        "fighter_a_id",
        "fighter_a_name",
        "fighter_b_id",
        "fighter_b_name",
        "target",
    ]
].copy()

test_output["probability_fighter_a_raw"] = p_test_raw
test_output["probability_fighter_a"] = p_test_final
test_output["probability_fighter_b"] = 1.0 - p_test_final

test_output["predicted_target"] = (
    p_test_final >= 0.5
).astype(int)

test_output["predicted_winner"] = np.where(
    test_output["predicted_target"].eq(1),
    test_output["fighter_a_name"],
    test_output["fighter_b_name"],
)

test_output["actual_winner"] = np.where(
    test_output["target"].eq(1),
    test_output["fighter_a_name"],
    test_output["fighter_b_name"],
)

test_output["correct_prediction"] = (
    test_output["predicted_target"] == test_output["target"]
)

test_output["confidence"] = np.maximum(
    test_output["probability_fighter_a"],
    test_output["probability_fighter_b"],
)

display(
    test_output[
        [
            "event_date",
            "event_name",
            "fighter_a_name",
            "fighter_b_name",
            "probability_fighter_a",
            "probability_fighter_b",
            "predicted_winner",
            "actual_winner",
            "correct_prediction",
        ]
    ].tail(30)
)


# Confusion matrix

In [ ]:
cm = confusion_matrix(
    y_test,
    test_output["predicted_target"],
)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Fighter B wins",
        "Fighter A wins",
    ],
)

display_cm.plot()
plt.title("Final test confusion matrix")
plt.show()


# Final probability reliability

In [ ]:
def reliability_table(
    y_true: np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 10,
) -> pd.DataFrame:
    y_true = np.asarray(y_true, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)

    bins = pd.cut(
        probabilities,
        bins=edges,
        include_lowest=True,
        right=True,
    )

    table = pd.DataFrame({
        "bin": bins,
        "probability": probabilities,
        "target": y_true,
    })

    result = (
        table.groupby("bin", observed=False)
        .agg(
            count=("target", "size"),
            mean_predicted_probability=("probability", "mean"),
            observed_win_rate=("target", "mean"),
        )
        .reset_index()
    )

    result["calibration_error"] = (
        result["observed_win_rate"]
        - result["mean_predicted_probability"]
    )

    return result


final_reliability = reliability_table(
    y_test.to_numpy(),
    p_test_final,
    n_bins=10,
)

display(final_reliability)


In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

valid = final_reliability["count"] > 0

plt.plot(
    final_reliability.loc[
        valid,
        "mean_predicted_probability",
    ],
    final_reliability.loc[
        valid,
        "observed_win_rate",
    ],
    marker="o",
    label="Final model",
)

plt.xlabel("Mean predicted probability")
plt.ylabel("Observed win rate")
plt.title("Final test reliability curve")
plt.legend()
plt.show()


# Accuracy by model confidence

In [ ]:
confidence_bins = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.90,
    1.00,
]

test_output["confidence_bin"] = pd.cut(
    test_output["confidence"],
    bins=confidence_bins,
    include_lowest=True,
)

confidence_summary = (
    test_output.groupby(
        "confidence_bin",
        observed=False,
    )
    .agg(
        fights=("fight_id", "size"),
        mean_confidence=("confidence", "mean"),
        accuracy=("correct_prediction", "mean"),
    )
    .reset_index()
)

display(confidence_summary)


# Save the final artifacts

In [ ]:
test_output.to_csv(
    FINAL_PREDICTIONS_PATH,
    index=False,
)

final_reliability.to_csv(
    FINAL_RELIABILITY_PATH,
    index=False,
)

final_metrics_payload = {
    "test_rows": int(len(test_idx)),
    "test_date_min": str(
        df.iloc[test_idx]["event_date"].min().date()
    ),
    "test_date_max": str(
        df.iloc[test_idx]["event_date"].max().date()
    ),
    "calibration_method": FROZEN_CALIBRATION_METHOD,
    "raw_catboost": raw_test_metrics,
    "final_probabilities": final_test_metrics,
    "frozen_catboost_params": FROZEN_CATBOOST_PARAMS,
}

with FINAL_METRICS_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_metrics_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )

if FINAL_CALIBRATOR is not None:
    with FINAL_CALIBRATOR_PATH.open("wb") as f:
        pickle.dump(FINAL_CALIBRATOR, f)

if oof_calibration_df is not None:
    oof_calibration_df.to_csv(
        FINAL_RESULTS_DIR
        / "final_calibration_oof_predictions.csv",
        index=False,
    )

print("Saved final predictions:", FINAL_PREDICTIONS_PATH)
print("Saved final metrics:", FINAL_METRICS_PATH)
print("Saved final model:", FINAL_MODEL_PATH)

if FINAL_CALIBRATOR is not None:
    print("Saved final calibrator:", FINAL_CALIBRATOR_PATH)


# Final report

In [ ]:
print("=" * 72)
print("FINAL UNTOUCHED TEST RESULT")
print("=" * 72)

print(
    "Test period:",
    df.iloc[test_idx]["event_date"].min().date(),
    "->",
    df.iloc[test_idx]["event_date"].max().date(),
)

print("Test fights:", f"{len(test_idx):,}")
print("Calibration:", FROZEN_CALIBRATION_METHOD)
print()

for metric, value in final_test_metrics.items():
    print(f"{metric:>12}: {value:.6f}")

print()
print(
    "The final test has now been opened. "
    "Do not tune the model against these results."
)


# Interpretation checklist

After running the notebook, I should document:

1. What was the exact untouched test period?
2. How many fights were evaluated?
3. What calibration policy had been frozen before opening the test?
4. What were the final Log Loss and Brier Score?
5. What was the final ROC AUC?
6. What was the final 0.5-threshold accuracy?
7. What was the final ECE?
8. Did the calibration policy improve or worsen the raw CatBoost probabilities on the final test?
9. Does the final reliability curve show major systematic overconfidence or underconfidence?
10. Does prediction accuracy generally increase with model confidence?
11. How do the final test metrics compare with the development-validation metrics?
12. Is the difference small enough to consider the development estimates representative?
13. What are the main remaining limitations of the model?

Regardless of the answers, the final test should remain an evaluation set rather than becoming a new validation set.